# Milestone 2: Exploratory Analysis + Feature Engineering

This notebook covers:
- Growth over time
- Genre and rating distributions by content type
- Country-level content contribution
- Feature engineering for duration categories and content origin


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.options.display.max_columns = 50
sns.set_theme(style="whitegrid")

DATA_PATH = Path("data/processed/netflix_titles_cleaned.csv")


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


## Growth Over Time


In [ ]:
growth = df.groupby("release_year").size().reset_index(name="count")
plt.figure(figsize=(10, 4))
sns.lineplot(data=growth, x="release_year", y="count")
plt.title("Total Content by Release Year")
plt.xlabel("Release Year")
plt.ylabel("Title Count")
plt.tight_layout()
plt.show()

growth_type = df.groupby(["release_year", "type"]).size().reset_index(name="count")
plt.figure(figsize=(10, 4))
sns.lineplot(data=growth_type, x="release_year", y="count", hue="type")
plt.title("Content Growth by Type")
plt.xlabel("Release Year")
plt.ylabel("Title Count")
plt.tight_layout()
plt.show()


## Genre Distribution (Movies vs TV Shows)


In [ ]:
genres = df.assign(genre=df["listed_in"].str.split("|")).explode("genre")
genre_counts = genres.groupby(["type", "genre"]).size().reset_index(name="count")
top_genres = (
    genre_counts.sort_values("count", ascending=False)
    .groupby("type")
    .head(10)
)

g = sns.catplot(
    data=top_genres,
    kind="bar",
    x="count",
    y="genre",
    col="type",
    height=5,
    aspect=0.9,
    sharex=False,
)
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Top 10 Genres by Content Type")
plt.show()


## Rating Distribution (Movies vs TV Shows)


In [ ]:
rating_counts = df.groupby(["type", "rating"]).size().reset_index(name="count")
rating_counts = rating_counts.sort_values(["type", "count"], ascending=[True, False])

g = sns.catplot(
    data=rating_counts,
    kind="bar",
    x="count",
    y="rating",
    col="type",
    height=5,
    aspect=0.9,
    sharex=False,
)
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Ratings by Content Type")
plt.show()


## Country-Level Content Contribution


In [ ]:
countries = df.assign(country=df["country"].str.split("|")).explode("country")
country_counts = countries.groupby("country").size().reset_index(name="count")
top_countries = country_counts.sort_values("count", ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(data=top_countries, x="count", y="country", palette="viridis")
plt.title("Top 10 Countries by Content Volume")
plt.xlabel("Title Count")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


## Feature Engineering


In [ ]:
sys.path.append(str(Path("src").resolve()))
from feature_engineering import add_feature_columns

featured = add_feature_columns(df)
featured[["duration", "duration_value", "duration_unit", "length_category", "content_origin"]].head()


In [ ]:
length_counts = featured.groupby(["type", "length_category"]).size().reset_index(name="count")
g = sns.catplot(
    data=length_counts,
    kind="bar",
    x="count",
    y="length_category",
    col="type",
    height=4.5,
    aspect=1.0,
    sharex=False,
)
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Length Category by Content Type")
plt.show()

origin_counts = featured.groupby(["type", "content_origin"]).size().reset_index(name="count")
g = sns.catplot(
    data=origin_counts,
    kind="bar",
    x="count",
    y="content_origin",
    col="type",
    height=4.5,
    aspect=1.0,
    sharex=False,
)
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Original vs Licensed by Content Type")
plt.show()


In [ ]:
FEATURED_PATH = Path("data/processed/netflix_titles_featured.csv")
featured.to_csv(FEATURED_PATH, index=False)
FEATURED_PATH
